In [ ]:
import numpy as np
from pathlib import Path
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

class FeatureEngineeringEngine:
    """
    Feature engineering specifically for 516-feature sign language data
    Data structure: [POSE(132) | HANDS(126) | VELOCITY(258)]
    """
    
    def __init__(self):
        # Feature indices
        self.POSE_START = 0
        self.POSE_END = 132
        self.HANDS_START = 132
        self.HANDS_END = 258
        self.VELOCITY_START = 258
        self.VELOCITY_END = 516
        
        # MediaPipe landmark indices for pose
        self.LEFT_SHOULDER = 11
        self.RIGHT_SHOULDER = 12
        self.LEFT_WRIST = 15
        self.RIGHT_WRIST = 16
        self.LEFT_HIP = 23
        self.RIGHT_HIP = 24
        self.NOSE = 0
        self.LEFT_ELBOW = 13
        self.RIGHT_ELBOW = 14
        
        # Hand fingertip indices (0-20)
        self.HAND_INDICES = {
            'wrist': 0,
            'thumb_tip': 4,
            'index_tip': 8,
            'middle_tip': 12,
            'ring_tip': 16,
            'pinky_tip': 20,
            'thumb_base': 1,
            'index_base': 5,
            'middle_base': 9,
            'ring_base': 13,
            'pinky_base': 17
        }
        
        # Joint chains for angle calculation
        self.JOINT_CHAINS = {
            'thumb': [1, 2, 3, 4],
            'index': [5, 6, 7, 8],
            'middle': [9, 10, 11, 12],
            'ring': [13, 14, 15, 16],
            'pinky': [17, 18, 19, 20]
        }
        
        # Store feature dimensions for tracking
        self.feature_dims = {}
    
    # ============================================================
    # EXTRACT COMPONENTS FROM 516 FEATURES
    # ============================================================
    
    def extract_pose_landmarks(self, features):
        """
        Extract pose landmarks (33, 4) from 516 features
        """
        return features[self.POSE_START:self.POSE_END].reshape(33, 4)
    
    def extract_left_hand(self, features):
        """
        Extract left hand landmarks (21, 3) from 516 features
        """
        hand_features = features[self.HANDS_START:self.HANDS_END]
        return hand_features[:63].reshape(21, 3)
    
    def extract_right_hand(self, features):
        """
        Extract right hand landmarks (21, 3) from 516 features
        """
        hand_features = features[self.HANDS_START:self.HANDS_END]
        return hand_features[63:].reshape(21, 3)
    
    def extract_velocity(self, features):
        """
        Extract velocity (258) from 516 features
        """
        return features[self.VELOCITY_START:self.VELOCITY_END]
    
    # ============================================================
    # 1. RELATIVE JOINT COORDINATES
    # ============================================================
    
    def compute_relative_coordinates(self, hand_landmarks):
        """
        Compute coordinates relative to wrist (21, 3) -> 63 features
        """
        wrist = hand_landmarks[0]
        relative = hand_landmarks - wrist
        return relative.flatten()
    
    def compute_finger_relative_positions(self, hand_landmarks):
        """
        Compute positions of each finger relative to its base
        (21, 3) -> 60 features (20 joints × 3)
        """
        features = []
        for finger_name, indices in self.JOINT_CHAINS.items():
            base = hand_landmarks[indices[0]]
            for idx in indices[1:]:
                relative = hand_landmarks[idx] - base
                features.extend(relative)
        return np.array(features)
    
    # ============================================================
    # 2. HAND-TO-BODY DISTANCES
    # ============================================================
    
    def compute_hand_to_body_distances(self, hand_landmarks, pose_landmarks):
        """
        Compute distances from hand center to body parts (7 distances)
        """
        hand_center = np.mean(hand_landmarks, axis=0)
        
        body_keypoints = {
            'nose': pose_landmarks[self.NOSE][:3],
            'left_shoulder': pose_landmarks[self.LEFT_SHOULDER][:3],
            'right_shoulder': pose_landmarks[self.RIGHT_SHOULDER][:3],
            'left_hip': pose_landmarks[self.LEFT_HIP][:3],
            'right_hip': pose_landmarks[self.RIGHT_HIP][:3],
            'left_elbow': pose_landmarks[self.LEFT_ELBOW][:3],
            'right_elbow': pose_landmarks[self.RIGHT_ELBOW][:3]
        }
        
        distances = []
        for name, pos in body_keypoints.items():
            dist = np.linalg.norm(hand_center - pos)
            distances.append(dist)
        
        return np.array(distances)
    
    def compute_wrist_to_shoulder_ratio(self, hand_landmarks, pose_landmarks, hand_type='left'):
        """
        Compute ratio of wrist distance to shoulder width (1 feature)
        """
        wrist = hand_landmarks[0]
        
        if hand_type == 'left':
            shoulder = pose_landmarks[self.LEFT_SHOULDER][:3]
        else:
            shoulder = pose_landmarks[self.RIGHT_SHOULDER][:3]
        
        wrist_to_shoulder = np.linalg.norm(wrist - shoulder)
        
        shoulder_width = np.linalg.norm(
            pose_landmarks[self.LEFT_SHOULDER][:3] - 
            pose_landmarks[self.RIGHT_SHOULDER][:3]
        )
        
        if shoulder_width > 0.001:
            ratio = wrist_to_shoulder / shoulder_width
        else:
            ratio = 0
        
        return np.array([ratio])
    
    # ============================================================
    # 3. JOINT ANGLES
    # ============================================================
    
    def compute_hand_angles(self, hand_landmarks):
        """
        Compute angles for all finger joints (15 angles)
        """
        angles = []
        
        for finger_name, indices in self.JOINT_CHAINS.items():
            if len(indices) >= 3:
                for i in range(len(indices) - 2):
                    p1 = hand_landmarks[indices[i]]
                    p2 = hand_landmarks[indices[i+1]]
                    p3 = hand_landmarks[indices[i+2]]
                    angle = self._compute_angle(p1, p2, p3)
                    angles.append(angle)
        
        return np.array(angles)
    
    def _compute_angle(self, p1, p2, p3):
        """Compute angle between three points in degrees"""
        v1 = p1 - p2
        v2 = p3 - p2
        
        v1_norm = np.linalg.norm(v1)
        v2_norm = np.linalg.norm(v2)
        
        if v1_norm > 0.001 and v2_norm > 0.001:
            cos_angle = np.dot(v1, v2) / (v1_norm * v2_norm)
            cos_angle = np.clip(cos_angle, -1.0, 1.0)
            angle = np.arccos(cos_angle) * 180 / np.pi
        else:
            angle = 0
        
        return angle
    
    def compute_hand_orientation_angles(self, hand_landmarks):
        """
        Compute overall hand orientation (2 features)
        """
        wrist = hand_landmarks[0]
        index_base = hand_landmarks[5]
        pinky_base = hand_landmarks[17]
        
        v1 = index_base - wrist
        v2 = pinky_base - wrist
        
        palm_normal = np.cross(v1, v2)
        if np.linalg.norm(palm_normal) > 0.001:
            palm_normal = palm_normal / np.linalg.norm(palm_normal)
        
        roll = np.arctan2(palm_normal[1], palm_normal[2]) * 180 / np.pi
        pitch = np.arctan2(-palm_normal[0], np.sqrt(palm_normal[1]**2 + palm_normal[2]**2)) * 180 / np.pi
        
        return np.array([roll, pitch])
    
    # ============================================================
    # 4. VELOCITY & ACCELERATION (from existing velocity)
    # ============================================================
    
    def extract_velocity_features(self, features_sequence):
        """
        Extract velocity from your existing 516 features
        Already computed, just extract it
        """
        T = features_sequence.shape[0]
        velocities = []
        
        for t in range(T):
            velocity = self.extract_velocity(features_sequence[t])
            velocities.append(velocity)
        
        return np.array(velocities)
    
    def compute_hand_speed(self, features_sequence):
        """
        Compute speed magnitude from existing velocity
        """
        T = features_sequence.shape[0]
        speeds = []
        
        for t in range(T):
            velocity = self.extract_velocity(features_sequence[t])
            # Velocity is flat vector, compute magnitude
            speed = np.linalg.norm(velocity)
            speeds.append(speed)
        
        return np.array(speeds).reshape(T, 1)
    
    def compute_hand_acceleration(self, features_sequence):
        """
        Compute acceleration from velocity (258 features)
        """
        T = features_sequence.shape[0]
        accelerations = []
        
        # First frame acceleration is zero
        accelerations.append(np.zeros(258))
        
        for t in range(1, T):
            vel_t = self.extract_velocity(features_sequence[t])
            vel_prev = self.extract_velocity(features_sequence[t-1])
            accel = vel_t - vel_prev
            accelerations.append(accel)
        
        return np.array(accelerations)
    
    def compute_acceleration_magnitude(self, features_sequence):
        """
        Compute magnitude of acceleration
        """
        T = features_sequence.shape[0]
        accel_mags = []
        
        accel_mags.append(0)  # First frame
        
        for t in range(1, T):
            vel_t = self.extract_velocity(features_sequence[t])
            vel_prev = self.extract_velocity(features_sequence[t-1])
            accel = vel_t - vel_prev
            accel_mag = np.linalg.norm(accel)
            accel_mags.append(accel_mag)
        
        return np.array(accel_mags).reshape(T, 1)
    
    # ============================================================
    # 5. MOTION DIRECTION
    # ============================================================
    
    def compute_motion_direction(self, features_sequence):
        """
        Compute motion direction from hand positions
        """
        T = features_sequence.shape[0]
        directions = []
        
        # Extract hand centers for each frame
        hand_centers = []
        for t in range(T):
            left_hand = self.extract_left_hand(features_sequence[t])
            right_hand = self.extract_right_hand(features_sequence[t])
            # Use left hand center for direction
            center = np.mean(left_hand, axis=0)
            hand_centers.append(center)
        
        hand_centers = np.array(hand_centers)
        
        # First frame direction is zero
        directions.append([0, 0])
        
        for t in range(1, T):
            direction = hand_centers[t] - hand_centers[t-1]
            direction_norm = np.linalg.norm(direction)
            
            if direction_norm > 0.001:
                direction = direction / direction_norm
                azimuth = np.arctan2(direction[1], direction[0]) * 180 / np.pi
                elevation = np.arctan2(direction[2], np.sqrt(direction[0]**2 + direction[1]**2)) * 180 / np.pi
                directions.append([azimuth, elevation])
            else:
                directions.append([0, 0])
        
        return np.array(directions)
    
    # ============================================================
    # COMPLETE FEATURE EXTRACTION
    # ============================================================
    
    def extract_all_features(self, features_sequence):
        """
        Extract all engineered features from a sequence of 516-feature frames
        """
        T = features_sequence.shape[0]
        
        # Initialize feature lists
        all_features = []
        
        # Store feature dimensions for tracking
        self.feature_dims = {}
        
        for t in range(T):
            features = features_sequence[t]
            
            # Extract components
            pose = self.extract_pose_landmarks(features)
            left_hand = self.extract_left_hand(features)
            right_hand = self.extract_right_hand(features)
            
            frame_features = []
            
            # 1. Relative joint coordinates (left hand: 63, right hand: 63)
            left_relative = self.compute_relative_coordinates(left_hand)
            right_relative = self.compute_relative_coordinates(right_hand)
            frame_features.append(left_relative)
            frame_features.append(right_relative)
            self.feature_dims['relative_coords'] = 126
            
            # 2. Hand-to-body distances (left: 7, right: 7)
            left_distances = self.compute_hand_to_body_distances(left_hand, pose)
            right_distances = self.compute_hand_to_body_distances(right_hand, pose)
            frame_features.append(left_distances)
            frame_features.append(right_distances)
            self.feature_dims['body_distances'] = 14
            
            # 3. Joint angles (left: 15, right: 15)
            left_angles = self.compute_hand_angles(left_hand)
            right_angles = self.compute_hand_angles(right_hand)
            frame_features.append(left_angles)
            frame_features.append(right_angles)
            self.feature_dims['joint_angles'] = 30
            
            # 4. Hand orientation angles (left: 2, right: 2)
            left_orientation = self.compute_hand_orientation_angles(left_hand)
            right_orientation = self.compute_hand_orientation_angles(right_hand)
            frame_features.append(left_orientation)
            frame_features.append(right_orientation)
            self.feature_dims['orientation'] = 4
            
            # 5. Wrist-to-shoulder ratio (left: 1, right: 1)
            left_ratio = self.compute_wrist_to_shoulder_ratio(left_hand, pose, 'left')
            right_ratio = self.compute_wrist_to_shoulder_ratio(right_hand, pose, 'right')
            frame_features.append(left_ratio)
            frame_features.append(right_ratio)
            self.feature_dims['shoulder_ratio'] = 2
            
            # Combine all frame features
            combined = np.concatenate(frame_features)
            all_features.append(combined)
        
        # Convert to numpy array
        all_features = np.array(all_features)  # (T, static_features)
        
        # 6. Velocity (already in data, extract it)
        velocity = self.extract_velocity_features(features_sequence)  # (T, 258)
        
        # 7. Speed (1 per frame)
        speed = self.compute_hand_speed(features_sequence)  # (T, 1)
        self.feature_dims['speed'] = 1
        
        # 8. Acceleration (from velocity) (T, 258)
        acceleration = self.compute_hand_acceleration(features_sequence)  # (T, 258)
        self.feature_dims['acceleration'] = 258
        
        # 9. Acceleration magnitude (T, 1)
        accel_mag = self.compute_acceleration_magnitude(features_sequence)  # (T, 1)
        self.feature_dims['accel_mag'] = 1
        
        # 10. Motion direction (T, 2)
        direction = self.compute_motion_direction(features_sequence)  # (T, 2)
        self.feature_dims['direction'] = 2
        
        # Combine all features
        final_features = np.concatenate([
            all_features,        # Static features
            velocity,            # Original velocity (258)
            speed,               # Speed magnitude
            acceleration,        # Acceleration (258)
            accel_mag,           # Acceleration magnitude
            direction            # Motion direction (2)
        ], axis=1)
        
        # Calculate total features
        self.feature_dims['total'] = final_features.shape[1]
        self.feature_dims['static'] = all_features.shape[1]
        
        return final_features
    
    def get_feature_summary(self):
        """Get summary of feature dimensions"""
        return self.feature_dims

# ============================================================
# PROCESS FILES
# ============================================================

def process_with_feature_engineering(input_dir, output_dir, classes=None, max_samples=None):
    """
    Process all files with feature engineering
    """
    input_dir = Path(input_dir)
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    
    if classes is None:
        class_folders = [d for d in input_dir.iterdir() if d.is_dir()]
    else:
        class_folders = [input_dir / cls for cls in classes if (input_dir / cls).exists()]
    
    print(f"Processing {len(class_folders)} classes...")
    
    engine = FeatureEngineeringEngine()
    total_files = 0
    
    for class_path in tqdm(class_folders, desc="Processing classes"):
        class_name = class_path.name
        out_class_path = output_dir / class_name
        out_class_path.mkdir(parents=True, exist_ok=True)
        
        npy_files = sorted(class_path.glob("*.npy"))
        
        if max_samples:
            npy_files = npy_files[:max_samples]
        
        for file_path in tqdm(npy_files, desc=f"  {class_name}", leave=False):
            try:
                data = np.load(file_path)
                if data.ndim == 1:
                    data = data.reshape(1, -1)
                
                # Verify data shape
                if data.shape[1] != 516:
                    print(f"  ⚠️ Skipping {file_path.name}: Expected 516 features, got {data.shape[1]}")
                    continue
                
                engineered = engine.extract_all_features(data)
                
                output_path = out_class_path / file_path.name
                np.save(output_path, engineered)
                total_files += 1
                
            except Exception as e:
                print(f"   Error processing {file_path.name}: {e}")
    
    # Print feature summary
    feature_dims = engine.get_feature_summary()
    print(f"\n Feature Engineering Summary:")
    print(f"  - Total features: {feature_dims.get('total', 0)}")
    print(f"  - Static features: {feature_dims.get('static', 0)}")
    print(f"  - Original velocity: 258")
    print(f"  - Added speed: 1")
    print(f"  - Added acceleration: 258")
    print(f"  - Added acceleration magnitude: 1")
    print(f"  - Added motion direction: 2")
    
    print(f"\n Feature engineering complete!")
    print(f"  - Files processed: {total_files}")

# ============================================================
# TEST
# ============================================================

def test_feature_engineering():
    """Test feature engineering with sample data"""
    
    print("=" * 60)
    print("TESTING FEATURE ENGINEERING (516 features)")
    print("=" * 60)
    
    # Create sample 516-feature data
    sample_data = np.random.randn(30, 516) * 0.1
    
    # Add some structure
    for t in range(30):
        # Pose
        pose = sample_data[t, :132].reshape(33, 4)
        pose[11, :3] = np.array([-0.2, 0.1, 0])
        pose[12, :3] = np.array([0.2, 0.1, 0])
        pose[23, :3] = np.array([-0.1, -0.2, 0])
        pose[24, :3] = np.array([0.1, -0.2, 0])
        sample_data[t, :132] = pose.flatten()
        
        # Left hand
        left_hand = sample_data[t, 132:195].reshape(21, 3)
        left_hand[0] = np.array([0, 0, 0])
        for idx in [4, 8, 12, 16, 20]:
            left_hand[idx] = np.random.randn(3) * 0.1 + np.array([0, 0.2, 0])
        sample_data[t, 132:195] = left_hand.flatten()
        
        # Right hand
        right_hand = sample_data[t, 195:258].reshape(21, 3)
        right_hand[0] = np.array([0.2, 0, 0])
        for idx in [4, 8, 12, 16, 20]:
            right_hand[idx] = np.random.randn(3) * 0.1 + np.array([0.2, 0.2, 0])
        sample_data[t, 195:258] = right_hand.flatten()
        
        # Velocity (simple)
        if t > 0:
            sample_data[t, 258:] = sample_data[t, :258] - sample_data[t-1, :258]
        else:
            sample_data[t, 258:] = 0
    
    engine = FeatureEngineeringEngine()
    engineered = engine.extract_all_features(sample_data)
    
    print(f"\n Results:")
    print(f"  - Input shape: {sample_data.shape}")
    print(f"  - Output shape: {engineered.shape}")
    print(f"  - Feature increase: {engineered.shape[1] / sample_data.shape[1]:.1f}x")
    
    feature_dims = engine.get_feature_summary()
    print(f"\n Feature Breakdown:")
    for name, dim in feature_dims.items():
        print(f"  - {name}: {dim}")

# ============================================================
# RUN
# ============================================================

if __name__ == "__main__":
    test_feature_engineering()
    
    print("\n" + "=" * 60)
    print("USAGE EXAMPLE:")
    print("=" * 60)
    
    # Process your data
    process_with_feature_engineering(
        input_dir="D:/uni/Intern-1-Project/ksl/landmarks_30frames_normalized_v2",
        output_dir="D:/uni/Intern-1-Project/ksl/landmarks_30frames_engineered_v2",
        classes=['កុំព្យូទ័រ','កៅអី','ក្ដារខៀន','ខ្មៅដៃ','ជ័រលុប','ដីស','តុ','ទឹកលុប','នាយករង','នាយិកា','បន្ទាត់','សៀវភៅ','ប៊ិកខៀវ','ហ្វឺតខ្មៅ',
        'ហ្វឺតក្រហម','ប៊ិក','ប៊ិកក្រហម','កាតាប','កាតាបស្ពាយក្រោយ','ហ្វឺតខៀវ'],
        max_samples=None
    )
    